# M0 · 04 — Elementwise ops & broadcasting

**Elementwise**: `a + b`, `a * b`, `a ** -0.5` apply to each element in parallel
(no loops). **Broadcasting**: PyTorch auto-stretches a smaller shape to match a
bigger one so they can combine.

The GPT relies on both: `x = tok_emb + pos_emb` (adds a per-position vector to
every batch), `* k.shape[-1]**-0.5` (scale all scores), and the mask add.

In [1]:
import torch
from m0_checks import check, check_tensor, TODO
torch.manual_seed(0)

## 1. Elementwise arithmetic

Add `10` to every element of `x` (broadcasting a scalar).

In [2]:
x = torch.tensor([1, 2, 3])
plus10 = x+10
plus10

tensor([11, 12, 13])

In [3]:
check('x + 10', plus10, torch.tensor([11, 12, 13]))

✅ x + 10


True

## 2. The `** -0.5` scaling from attention

`q @ kᵀ * head_size**-0.5` divides scores by √head_size. `y ** -0.5` is `1/√y`.

Scale the vector `s` by `4 ** -0.5` (i.e. divide by 2).

In [4]:
s = torch.tensor([2.0, 4.0, 8.0])
scaled = s*(4**-0.5)
scaled

tensor([1., 2., 4.])

In [5]:
check('scaled by 1/2', scaled, torch.tensor([1.0, 2.0, 4.0]))

✅ scaled by 1/2


True

## 3. Broadcasting rule (the mental model)

Align shapes from the **right**. Two axes are compatible if they're equal or one
of them is 1 (the size-1 axis gets stretched). Example: `(2,3) + (3,)` → the
`(3,)` is treated as `(1,3)` and copied across both rows.

Add the row vector `r = [10, 20, 30]` to **every row** of the 2×3 matrix `m`.

In [6]:
m = torch.tensor([[0, 0, 0], [1, 1, 1]]) # 2,3
r = torch.tensor([10, 20, 30]) # 1,3
out = m+r
out

tensor([[10, 20, 30],
        [11, 21, 31]])

In [7]:
check('row broadcast', out, torch.tensor([[10, 20, 30], [11, 21, 31]]))

✅ row broadcast


True

## 4. This *is* `tok_emb + pos_emb`

`tok_emb` is `(B, T, C)`; `pos_emb` is `(T, C)` — the *same* position vectors for
every batch. Broadcasting stretches `pos_emb` across the batch axis so each of the
B sequences gets the same positional signal added.

Add `pos` `(T=2, C=3)` to `tok` `(B=2, T=2, C=3)` — result stays `(2, 2, 3)`.

In [8]:
tok = torch.zeros(2, 2, 3) #
pos = torch.tensor([[1., 1., 1.], [2., 2., 2.]])  # (T=2, C=3)
x = tok+pos
x

tensor([[[1., 1., 1.],
         [2., 2., 2.]],

        [[1., 1., 1.],
         [2., 2., 2.]]])

In [9]:
check_tensor('sum shape (2,2,3)', x, shape=(2, 2, 3))
check('pos added to every batch', x, torch.tensor(
    [[[1., 1., 1.], [2., 2., 2.]], [[1., 1., 1.], [2., 2., 2.]]]))

✅ sum shape (2,2,3)
✅ pos added to every batch


True

## 5. When broadcasting FAILS (good to see once)

Shapes `(2,3)` and `(2,)` are **not** compatible (right-aligned: 3 vs 2, neither
is 1). Run this to see the error message — recognising it saves debugging time.

In [10]:
try:
    torch.zeros(2, 3) + torch.tensor([1, 2])
    print('no error (unexpected)')
except RuntimeError as e:
    print('RuntimeError (expected):', str(e).splitlines()[0])

RuntimeError (expected): The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1


## ✅ Recap

- Elementwise ops (`+`, `*`, `**-0.5`) act per-element, in parallel.
- Broadcasting right-aligns shapes; size-1 (or missing) axes stretch.
- `tok_emb + pos_emb` broadcasts the `(T,C)` positions across the batch.
- Incompatible shapes raise a `RuntimeError` — read it, fix the shapes.

Next: **05 — matrix multiply** (`@`), the heart of attention.